# Train Segment Labels On CLAP Embeddings

This notebook reads labels from the independent annotation tables, loads segment-level CLAP embeddings from Chroma, performs a stratified split, and trains a small multi-label Keras MLP.

It does not require the main API or annotation API to be running.

## Configuration

Set these environment variables before running, or edit the values below:

- `STREETPARADE_DB`: SQLite DB containing `annotation_campaign`, `annotation_assignments`, and `sample_embeddings`.
- `STREETPARADE_CHROMA_DIR`: Chroma persistence directory.
- `ANNOTATION_CAMPAIGN_ID`: campaign ID to train on.

In [ ]:
from __future__ import annotations

import json
import os
import sqlite3
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

DB_PATH = Path(os.environ.get("STREETPARADE_DB", "data/streetparade_embeddings.sqlite3"))
CHROMA_DIR = Path(os.environ.get("STREETPARADE_CHROMA_DIR", "chroma"))
CAMPAIGN_ID = int(os.environ.get("ANNOTATION_CAMPAIGN_ID", "1"))
COLLECTION_NAME = os.environ.get("STREETPARADE_CHROMA_COLLECTION", "track_embeddings")
RANDOM_STATE = 42

print({
    "db": str(DB_PATH),
    "chroma": str(CHROMA_DIR),
    "campaign_id": CAMPAIGN_ID,
    "collection": COLLECTION_NAME,
})

## Load Label Catalog And Labeled Segments

The model is multi-label. Each target row is a multi-hot vector over all labels in the selected `annotation_campaign`.

In [ ]:
def connect(db_path: Path) -> sqlite3.Connection:
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    return conn


def load_label_catalog(db_path: Path, campaign_id: int) -> pd.DataFrame:
    query = """
        SELECT
            labels.id AS label_id,
            labels.name AS label_name,
            label_sets.id AS label_set_id,
            label_sets.name AS label_set_name,
            label_sets.sort_order AS label_set_sort_order,
            labels.sort_order AS label_sort_order
        FROM annotation_labels labels
        JOIN annotation_label_sets label_sets ON label_sets.id = labels.label_set_id
        WHERE label_sets.annotation_campaign_id = ?
          AND labels.is_active = 1
        ORDER BY label_sets.sort_order, label_sets.name, labels.sort_order, labels.name
    """
    with connect(db_path) as conn:
        return pd.DataFrame([dict(row) for row in conn.execute(query, (campaign_id,))])


def load_labeled_segments(db_path: Path, campaign_id: int) -> pd.DataFrame:
    query = """
        SELECT
            items.id AS annotation_item_id,
            items.track_id,
            items.track_sample_id AS sound_segment_id,
            samples.chunk_index,
            samples.start_seconds AS start_time,
            samples.start_seconds + samples.duration_seconds AS end_time,
            tracks.url AS track_url,
            tracks.path AS track_path,
            artists.name AS artist_name,
            latest_embedding.vector_id,
            latest_embedding.embedding_dim,
            labels.id AS label_id,
            labels.name AS label_name,
            label_sets.id AS label_set_id,
            label_sets.name AS label_set_name
        FROM annotation_items items
        JOIN track_samples samples ON samples.id = items.track_sample_id
        JOIN tracks ON tracks.id = items.track_id
        LEFT JOIN artists ON artists.id = tracks.artist_id
        JOIN annotation_assignments assignments
          ON assignments.annotation_campaign_id = items.annotation_campaign_id
         AND assignments.track_sample_id = items.track_sample_id
        JOIN annotation_labels labels ON labels.id = assignments.label_id
        JOIN annotation_label_sets label_sets ON label_sets.id = assignments.label_set_id
        LEFT JOIN sample_embeddings latest_embedding ON latest_embedding.id = (
            SELECT latest.id
            FROM sample_embeddings latest
            WHERE latest.track_sample_id = items.track_sample_id
            ORDER BY latest.embedded_at DESC, latest.id DESC
            LIMIT 1
        )
        WHERE items.annotation_campaign_id = ?
        ORDER BY items.track_id, samples.chunk_index, label_sets.name, labels.name
    """
    with connect(db_path) as conn:
        return pd.DataFrame([dict(row) for row in conn.execute(query, (campaign_id,))])


label_catalog = load_label_catalog(DB_PATH, CAMPAIGN_ID)
raw_labels = load_labeled_segments(DB_PATH, CAMPAIGN_ID)

print(f"labels: {len(label_catalog)}")
print(f"labeled assignment rows: {len(raw_labels)}")
display(label_catalog.head())
display(raw_labels.head())

In [ ]:
if label_catalog.empty:
    raise ValueError(f"No active labels found for annotation_campaign {CAMPAIGN_ID}")
if raw_labels.empty:
    raise ValueError(f"No labeled segments found for annotation_campaign {CAMPAIGN_ID}. Check annotation_assignments and annotation_items.")

label_ids = label_catalog["label_id"].astype(int).tolist()
label_names = (label_catalog["label_set_name"] + "/" + label_catalog["label_name"]).tolist()
label_to_index = {label_id: idx for idx, label_id in enumerate(label_ids)}

segment_rows = []
for sound_segment_id, group in raw_labels.groupby("sound_segment_id", sort=False):
    first = group.iloc[0].to_dict()
    assigned_label_ids = sorted({int(value) for value in group["label_id"] if int(value) in label_to_index})
    if not assigned_label_ids:
        continue
    first["label_ids"] = assigned_label_ids
    first["label_signature"] = "|".join(str(value) for value in assigned_label_ids)
    segment_rows.append(first)

segments = pd.DataFrame(segment_rows).reset_index(drop=True)
segments_with_embeddings = segments[segments["vector_id"].notna()].reset_index(drop=True)
segments_missing_embeddings = segments[segments["vector_id"].isna()].reset_index(drop=True)

print(f"unique labeled segments: {len(segments)}")
print(f"labeled segments with embeddings: {len(segments_with_embeddings)}")
print(f"labeled segments missing embeddings: {len(segments_missing_embeddings)}")
if segments_with_embeddings.empty:
    raise ValueError("Labels were loaded, but no labeled segments have sample_embeddings. Compute segment embeddings before training.")

display(segments[["track_id", "sound_segment_id", "start_time", "end_time", "artist_name", "label_signature", "vector_id"]].head())
if not segments_missing_embeddings.empty:
    display(segments_missing_embeddings[["track_id", "sound_segment_id", "start_time", "end_time", "artist_name", "label_signature"]].head())

## Load CLAP Segment Embeddings From Chroma

In [ ]:
import chromadb

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_or_create_collection(name=COLLECTION_NAME, metadata={"hnsw:space": "cosine"})

trainable_segments = segments_with_embeddings.copy()
vector_ids = trainable_segments["vector_id"].astype(str).tolist()
chroma_result = collection.get(ids=vector_ids, include=["embeddings"])
embedding_by_id = {vector_id: embedding for vector_id, embedding in zip(chroma_result["ids"], chroma_result["embeddings"])}

missing = [vector_id for vector_id in vector_ids if vector_id not in embedding_by_id]
if missing:
    raise ValueError(f"Missing {len(missing)} embeddings from Chroma. Example: {missing[:3]}")

X = np.asarray([embedding_by_id[vector_id] for vector_id in vector_ids], dtype=np.float32)
Y = np.zeros((len(trainable_segments), len(label_ids)), dtype=np.float32)
for row_idx, assigned_label_ids in enumerate(trainable_segments["label_ids"]):
    for label_id in assigned_label_ids:
        Y[row_idx, label_to_index[int(label_id)]] = 1.0

print("X", X.shape, X.dtype)
print("Y", Y.shape, Y.dtype)
print("positive labels per segment", Counter(Y.sum(axis=1).astype(int)))

## Stratified Train / Validation / Test Split

For multi-label data, exact iterative stratification requires an extra dependency. This notebook uses exact label-signature stratification when each signature has enough samples. If sparse signatures make that impossible, it falls back to a deterministic random split.

In [ ]:
def stratified_or_random_split(indices, signatures, test_size, random_state):
    counts = Counter(signatures)
    can_stratify = len(counts) > 1 and min(counts.values()) >= 2
    stratify = signatures if can_stratify else None
    return train_test_split(indices, test_size=test_size, random_state=random_state, stratify=stratify)


all_indices = np.arange(len(trainable_segments))
signatures = trainable_segments["label_signature"].astype(str).to_numpy()
train_val_idx, test_idx = stratified_or_random_split(all_indices, signatures, test_size=0.15, random_state=RANDOM_STATE)
train_val_signatures = signatures[train_val_idx]
train_idx, val_idx = stratified_or_random_split(train_val_idx, train_val_signatures, test_size=0.1765, random_state=RANDOM_STATE)

X_train, Y_train = X[train_idx], Y[train_idx]
X_val, Y_val = X[val_idx], Y[val_idx]
X_test, Y_test = X[test_idx], Y[test_idx]

print({"train": len(train_idx), "val": len(val_idx), "test": len(test_idx)})
print("train signatures", Counter(signatures[train_idx]).most_common(8))
print("val signatures", Counter(signatures[val_idx]).most_common(8))
print("test signatures", Counter(signatures[test_idx]).most_common(8))

## Normalize Features

CLAP vectors are usually already well-behaved, but standardizing from the training split makes the MLP easier to train.

In [ ]:
feature_mean = X_train.mean(axis=0, keepdims=True)
feature_std = X_train.std(axis=0, keepdims=True)
feature_std = np.where(feature_std < 1e-6, 1.0, feature_std)

X_train_n = (X_train - feature_mean) / feature_std
X_val_n = (X_val - feature_mean) / feature_std
X_test_n = (X_test - feature_mean) / feature_std

## Train A Small Multi-Label MLP

In [ ]:
import tensorflow as tf
from tensorflow import keras

tf.keras.utils.set_random_seed(RANDOM_STATE)

embedding_dim = X_train_n.shape[1]
num_labels = Y_train.shape[1]

model = keras.Sequential([
    keras.layers.Input(shape=(embedding_dim,)),
    keras.layers.Dense(256, activation="relu"),
    keras.layers.Dropout(0.25),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.20),
    keras.layers.Dense(num_labels, activation="sigmoid"),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[keras.metrics.BinaryAccuracy(name="binary_accuracy"), keras.metrics.AUC(name="auc", multi_label=True)],
)

model.summary()

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5),
]

history = model.fit(
    X_train_n,
    Y_train,
    validation_data=(X_val_n, Y_val),
    epochs=100,
    batch_size=min(64, max(8, len(X_train_n))),
    callbacks=callbacks,
    verbose=1,
)

## Evaluate And Inspect Predictions

In [ ]:
test_metrics = model.evaluate(X_test_n, Y_test, verbose=0, return_dict=True)
test_metrics

In [ ]:
threshold = 0.5
probs = model.predict(X_test_n)
preds = (probs >= threshold).astype(int)

inspection = trainable_segments.iloc[test_idx].copy().reset_index(drop=True)
inspection["true_labels"] = [[label_names[i] for i, value in enumerate(row) if value > 0] for row in Y_test]
inspection["predicted_labels"] = [[label_names[i] for i, value in enumerate(row) if value > 0] for row in preds]
inspection["top_scores"] = [
    [(label_names[i], float(score)) for i, score in sorted(enumerate(row), key=lambda item: item[1], reverse=True)[:5]]
    for row in probs
]

display(inspection[["track_id", "sound_segment_id", "artist_name", "start_time", "end_time", "true_labels", "predicted_labels", "top_scores"]].head(20))

## Save Model And Metadata

In [ ]:
output_dir = Path("outputs") / f"annotation_campaign_{CAMPAIGN_ID}"
output_dir.mkdir(parents=True, exist_ok=True)

model.save(output_dir / "segment_label_mlp.keras")
np.savez_compressed(output_dir / "feature_normalization.npz", mean=feature_mean.astype(np.float32), std=feature_std.astype(np.float32))

metadata = {
    "annotation_campaign_id": CAMPAIGN_ID,
    "db_path": str(DB_PATH),
    "chroma_dir": str(CHROMA_DIR),
    "collection_name": COLLECTION_NAME,
    "label_ids": label_ids,
    "label_names": label_names,
    "embedding_dim": int(embedding_dim),
    "num_labeled_segments": int(len(segments)),
    "num_labeled_segments_with_embeddings": int(len(trainable_segments)),
    "num_labeled_segments_missing_embeddings": int(len(segments_missing_embeddings)),
    "num_train": int(len(train_idx)),
    "num_val": int(len(val_idx)),
    "num_test": int(len(test_idx)),
    "test_metrics": {key: float(value) for key, value in test_metrics.items()},
}
with (output_dir / "metadata.json").open("w", encoding="utf-8") as handle:
    json.dump(metadata, handle, indent=2, ensure_ascii=False)

print(f"saved to {output_dir}")